# SentiMind — max-performance classifier fine-tune

Goal: squeeze the most out of the classifier WITHOUT changing architecture — same distilroberta, **max_length=512**, tuned as hard as a free T4 will allow, evaluated with **sliding-window chunked inference** so long journals are actually read.

**You must upload** the 6 files from `backend/data/finetune/` (Colab Files sidebar → Upload):

`train.csv` · `val.csv` · `test.csv` · `adversarial.csv` · `labels.json` · `sklearn_baseline.json`

Order: run cells top to bottom. ~45–90 min on a free T4. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
!pip -q install "transformers>=4.44,<5" datasets accelerate evaluate scikit-learn

## 1. Load the splits
`test.csv` is the SAME fold-0 holdout v8 was measured on — the only apples-to-apples comparison available. Never train on it.

In [ ]:

import json, math
import numpy as np
import pandas as pd

LABEL2ID = json.load(open("labels.json"))
ORDER = [k for k, _ in sorted(LABEL2ID.items(), key=lambda kv: kv[1])]
print("classes:", ORDER)

for f in ["train.csv", "val.csv", "test.csv", "adversarial.csv"]:
    df = pd.read_csv(f)
    print(f, df.shape)

train = pd.read_csv("train.csv")
val   = pd.read_csv("val.csv")
test  = pd.read_csv("test.csv")
adv   = pd.read_csv("adversarial.csv")


## 2. Tokenize at max_length=512
v8 used 256. 512 covers ~5k-word posts (~85–90% of them fit entirely).

In [ ]:

from datasets import Dataset
from transformers import AutoTokenizer

MODEL_ID = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def src(df):
    return Dataset.from_pandas(df[["text", "label"]], preserve_index=False)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding=False)

train_ds = src(train).map(tok, batched=True).rename_column("label", "labels").remove_columns("text")
val_ds   = src(val).map(tok, batched=True).rename_column("label", "labels").remove_columns("text")
print(train_ds, val_ds)


## 3. Full fine-tune, fp16, tuned harder than v8
batch 16 + grad-accum 2 (effective 32), lr 2e-5 with cosine decay, 4 epochs, best-on-macro-F1 checkpoint.

In [ ]:

from transformers import (
    AutoModelForSequenceClassification, DataCollatorWithPadding,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
import evaluate, numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=5, id2label={i: k for k, i in LABEL2ID.items()})
f1_macro = evaluate.load("f1"); acc = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
            "f1_macro": f1_macro.compute(predictions=preds, references=p.label_ids,
                                         average="macro")["f1"]}

args = TrainingArguments(
    output_dir="./v9",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    seed=42,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
print(trainer.evaluate(val_ds))


## 4. Scoring — three ways
(a) plain; (b) chunked sliding window — the decisive test for long journals; (c) adversarial.

In [ ]:
import torch, numpy as np
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, classification_report

def batched_predict(texts, window, step, max_tokens=512, bs=16):
    probs = []
    for i in range(0, len(texts), bs):
        for t in texts[i:i+bs]:
            ids = tokenizer(t, truncation=True, max_length=max_tokens)["input_ids"]
            if len(ids) <= max_tokens:
                batch = tokenizer(t, truncation=True, max_length=max_tokens,
                                  return_tensors="pt").to(model.device)
                with torch.no_grad():
                    logits = model(**ids).logits if False else model(**batch).logits
                probs.append(F.softmax(logits, dim=-1)[0].cpu())
            else:
                chunks = []
                for start in range(0, len(ids), step):
                    piece = ids[start:start+window]
                    batch = {"input_ids": torch.tensor([piece]).to(model.device),
                             "attention_mask": torch.ones((1, len(piece)), dtype=torch.long).to(model.device)}
                    with torch.no_grad():
                        logits = model(**batch).logits
                    chunks.append(F.softmax(logits, dim=-1)[0].cpu())
                    if start + window >= len(ids):
                        break
                probs.append(torch.stack(chunks).mean(0))
    probs = torch.stack(probs)
    return probs.argmax(-1).numpy(), probs.numpy()

def predict_plain(df, bs=16):
    return batched_predict(list(df["text"]), window=512, step=512, bs=bs)

def predict_chunked(df, window=384, stride=128, bs=16):
    return batched_predict(list(df["text"]), window=window, step=stride, bs=bs)

for name, fn in [("plain 512", predict_plain), ("chunked sliding window", predict_chunked)]:
    pred, _ = fn(test)
    print("=", name)
    print("accuracy:", round(accuracy_score(test["label"], pred), 4))
    print("macro F1:", round(f1_score(test["label"], pred, average="macro"), 4))
    print(classification_report(test["label"], pred, target_names=ORDER))


In [ ]:

# Adversarial hard negatives
pred = predict_plain(adv)
print("adversarial (plain):", int((pred == adv["label"].to_numpy()).sum()), "/", len(adv))
pred = predict_chunked(adv)
print("adversarial (chunked):", int((pred == adv["label"].to_numpy()).sum()), "/", len(adv))

# Length buckets on the held-out fold
test = test.copy(); test["n_words"] = test["text"].str.split().str.len()
pred = predict_chunked(test)
for lo, hi, lab in [(0,25,"<25"),(25,60,"25-60"),(60,120,"60-120"),(120,260,"120-260"),(260,10**9,">260")]:
    m = test["n_words"].ge(lo) & test["n_words"].lt(hi)
    if m.sum():
        print(f"{lab:<8} n={int(m.sum()):>5}  acc={accuracy_score(test.loc[m,'label'], pred[m]):.3f}  macroF1={f1_score(test.loc[m,'label'], pred[m], average='macro'):.3f}")


## 5. Export INT8 ONNX (only if it wins)
If the table beats v8's 0.722 macro F1, export here and plug the artifact in place of v8's `model_int8.onnx`.
Skip this cell if it doesn't — the test set would be wasted.

In [ ]:

import torch
model.eval(); model.to("cpu")
opt_in = False  # set True only if the numbers won
if opt_in:
    dummy = tokenizer("sample text", return_tensors="pt")
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]),
        "model.onnx", input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={"input_ids": {0:"batch", 1:"seq"},
                      "attention_mask": {0:"batch", 1:"seq"},
                      "logits": {0:"batch"}},
        opset_version=17, dynamo=False)
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic("model.onnx", "model_int8.onnx", weight_type=QuantType.QInt8)
    print("exported", __import__("os").path.getsize("model_int8.onnx") / 1e6, "MB")


## 6. Verdict table

| Metric | v7 (sklearn) | v8 (distilroberta-256) | v9 (this run) |
|---|---|---|---|
| Accuracy | 0.664 | 0.714 | ? |
| Macro F1 | 0.645 | 0.722 | ? |
| Suicidal recall | 0.753 | 0.753 | ? |
| >260-word F1 | 0.614 | 0.586 | ? |

**Ship it only if v9 beats both v7 and v8 on macro F1 AND the >260 bucket without collapsing Suicidal recall.**